# Attention Output Decomposition

Decompose attention head outputs: per-source contributions, vocabulary
alignment, logit effects, attention-weighted value analysis, and
output direction stability.

## Why This Matters

Composition attention output decomposition measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_output_decomposition import (
    per_source_output_decomposition, output_vocabulary_alignment,
    head_logit_effect_profile, attention_weighted_value_analysis,
    head_output_direction_stability,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Per-Source Output Decomposition

Break down a head's output by which source token contributed what.

In [ ]:
result = per_source_output_decomposition(model, tokens, layer=0, head=0, position=-1)
print(f"Head L{result['layer']}H{result['head']} at position {result['position']}:\n")
for s in result['per_source']:
    bar = '█' * int(s['attention_weight'] * 50)
    print(f"  src {s['source_position']} (tok {s['source_token']:3d}): "
          f"attn={s['attention_weight']:.3f} norm={s['output_norm']:.4f} {bar}")

## Vocabulary Alignment

What vocabulary tokens does this head's output promote/suppress?

In [ ]:
result = output_vocabulary_alignment(model, tokens, layer=0, head=0, top_k=5)
print(f"Output norm: {result['output_norm']:.4f}\n")
print('Promoted:')
for t in result['promoted_tokens']:
    print(f"  token {t['token']:3d}: logit={t['logit']:+.4f}")
print('\nSuppressed:')
for t in result['suppressed_tokens']:
    print(f"  token {t['token']:3d}: logit={t['logit']:+.4f}")

## Head Logit Effect Profile

How does this head affect the predicted token logit at each position?

In [ ]:
result = head_logit_effect_profile(model, tokens, layer=0, head=0)
print(f"Mean logit contribution: {result['mean_logit_contribution']:.4f}\n")
for p in result['per_position']:
    sign = '+' if p['logit_contribution'] > 0 else '-'
    print(f"  Pos {p['position']}: {sign}{abs(p['logit_contribution']):.4f} "
          f"(target={p['target_token']}, norm={p['output_norm']:.4f})")

## Value Analysis

Analyze the attention-weighted value mixture at each position.

In [ ]:
result = attention_weighted_value_analysis(model, tokens, layer=0, head=0)
print(f"Mean entropy: {result['mean_entropy']:.4f}\n")
for p in result['per_position']:
    print(f"  Pos {p['position']}: entropy={p['attention_entropy']:.4f}, "
          f"max_attn={p['max_attention']:.3f}, "
          f"sources={p['n_attended_sources']}, val_norm={p['weighted_value_norm']:.4f}")

## Direction Stability

Does the head output in a consistent direction across positions?

In [ ]:
result = head_output_direction_stability(model, tokens, layer=0, head=0)
stable = 'STABLE' if result['is_stable'] else 'variable'
print(f"Mean pairwise similarity: {result['mean_pairwise_similarity']:.4f} [{stable}]\n")
for p in result['per_position']:
    print(f"  Pos {p['position']}: align_to_mean={p['alignment_to_mean']:+.4f}, "
          f"norm={p['output_norm']:.4f}")